# CP-APR: Alternating Poisson Regression for fitting CP to sparse count data

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

The function `cp_apr` computes a nonnegative CP decomposition of a tensor of count data using alternating Poisson regression. It assumes that each entry of the data tensor is an independent Poisson draw whose mean is given by the corresponding entry of a low-rank (CP) model with nonnegative factors, and it computes a maximum likelihood estimate of the model by alternating over the factor matrices. The input $\mathcal{X}$ can be a `tensor` or a `sptensor`. The output CP model is a `ktensor`.

## Problem setup

We follow the general procedure outlined by E. C. Chi and T. G. Kolda, On Tensors, Sparsity, and Nonnegative Factorizations, arXiv:1112.2414 [math.NA], December 2011 (http://arxiv.org/abs/1112.2414) to create a sparse count tensor. First, we create a nonnegative `ktensor` whose factor matrices have a few large entries in each column, and scale it so that the expected total number of events is 1000. Then, we sample each entry of the data tensor from a Poisson distribution whose mean is the corresponding entry of the `ktensor`.

In [ ]:
from __future__ import annotations

import os
import sys

import numpy as np

import pyttb as ttb

In [ ]:
# Pick the shape and rank
sz = (10, 8, 6)
R = 5

# Generate factor matrices with a few large entries in each column
# this will be the basis of our solution.
np.random.seed(0)  # Set seed for reproducibility
A = []
for n in range(len(sz)):
    A.append(np.random.uniform(size=(sz[n], R)))
    for r in range(R):
        p = np.random.permutation(sz[n])
        nbig = round((1 / R) * sz[n])
        A[-1][p[0:nbig], r] *= 100
weights = np.random.uniform(size=(R,))
S = ttb.ktensor(A, weights)
S.normalize(sort=True, normtype=1)
S.weights = S.weights * (1000 / S.weights.sum())  # expected total number of events

# Sample the counts from a Poisson distribution with mean given by the model
X = ttb.tensor(np.random.poisson(S.to_tensor().data)).to_sptensor()
print(f"X has {X.nnz} nonzeros out of {np.prod(X.shape)} entries")

## Basic usage

`cp_apr` takes a (sparse) count tensor and the rank of the model. Here we cut the solve off early with `stoptime` to keep the demo short.

In [ ]:
# Compute a solution
short_tutorial = 2  # Cut off solve early for demo
M, Minit, output = ttb.cp_apr(X, R, printitn=10, stoptime=short_tutorial)

## Main options

The `algorithm` option selects the solver: `"mu"` (multiplicative update, the default), `"pdnr"` (projected damped Newton for row subproblems), or `"pqnr"` (projected quasi-Newton for row subproblems). Other common options are `stoptol` (tolerance on the KKT violation), `maxiters`, `stoptime`, and `printitn`. Below we compare the first two on our sample problem.

In [ ]:
for algorithm in ["mu", "pdnr"]:
    M_alg, _, output_alg = ttb.cp_apr(
        X, R, algorithm=algorithm, maxiters=10, stoptime=2, printitn=0
    )
    print(f"{algorithm}: final KKT violation {output_alg['kktViolations'][-1]:.2e}")

## Interpreting the output

`cp_apr` returns the model `M` as a `ktensor`, the initial guess `Minit`, and a dictionary `output` with information about the run, including the parameters used (`params`), the KKT violation after each outer iteration (`kktViolations`), and the total solve time (`totalTime`).

In [ ]:
print(f"Model: {M.ncomponents} components with shape {M.shape}")
print(f"Keys in output: {sorted(output.keys())}")
print(
    f"Final KKT violation after {len(output['kktViolations'])} outer "
    f"iterations: {output['kktViolations'][-1]:.2e}"
)

## Worked example

Because we generated the data from a known `ktensor` `S`, we can measure how well the factorization recovers it with the `score` method (1 is a perfect match).

In [ ]:
M_final, _, _ = ttb.cp_apr(X, R, maxiters=100, stoptime=5, printitn=0)
print(f"Score against the true solution: {M_final.score(S)[0]:.2f}")